# Imports

In [0]:
from pyspark.sql.functions import trim, when, length, lit, col, row_number, lower as lower_spark, concat_ws, coalesce, current_timestamp, sha2
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
from pyspark.sql.functions import sum as sum_spark, lower as lower_spark, upper as upper_spark, countDistinct, first, dense_rank, floor

In [0]:
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

BRONZE_RACE_RESULTS_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.race_results"
)

SILVER_RACE_RESULTS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.race_results"
)

SILVER_RACES_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.races"
)

SILVER_DRIVERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.drivers"
)

SILVER_CONSTRUCTORS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.constructors"
)

# Metodos

In [0]:
def clean_string(column_name: str):
    value = trim(col(column_name))

    return (
        when(length(value) == 0, lit(None).cast("string"))
         .otherwise(value)
    )

In [0]:
def ns_to_ms(column_name: str):
    return (
        when(
            col(column_name).isNotNull(),
            (col(column_name) / 1_000_000).cast("long")
        )
    )

In [0]:
def get_latest_race_results_snapshot():
    bronze_df = spark.table(
        BRONZE_RACE_RESULTS_TABLE
    )

    snapshot_window = (
        Window
        .partitionBy(
            "season",
            "round"
        )
        .orderBy(
            col("_source_file_modification_time")
                .desc_nulls_last(),

            col("_ingested_at")
                .desc_nulls_last()
        )
    )

    return (
        bronze_df
        .withColumn(
            "_snapshot_rank",
            dense_rank().over(snapshot_window)
        )
        .filter(
            col("_snapshot_rank") == 1
        )
        .drop("_snapshot_rank")
    )

In [0]:
def transform_race_results(df):
    result_time_ms = ns_to_ms("Time")

    race_results_df = (
        df
        .select(
            # ----------------------------------------------
            # Race
            # ----------------------------------------------

            col("season")
                .cast("int")
                .alias("season"),

            col("round")
                .cast("int")
                .alias("round"),

            # ----------------------------------------------
            # Driver / Constructor
            # ----------------------------------------------

            lower_spark(
                clean_string("DriverId")
            ).alias("driver_id"),

            clean_string("DriverNumber")
                .alias("driver_number"),

            lower_spark(
                clean_string("TeamId")
            ).alias("constructor_id"),

            # ----------------------------------------------
            # Result
            # ----------------------------------------------

            col("Position")
                .cast("int")
                .alias("finish_position"),

            upper_spark(
                clean_string("ClassifiedPosition")
            ).alias("classified_position"),

            # Grid 0 es válido: pit lane.
            # Valores negativos no representan
            # una posición válida.
            when(
                col("GridPosition").isNull(),
                lit(None).cast("int")
            )
            .when(
                col("GridPosition") < 0,
                lit(None).cast("int")
            )
            .otherwise(
                col("GridPosition").cast("int")
            )
            .alias("grid_position"),

            # ----------------------------------------------
            # Time
            # ----------------------------------------------

            when(
                col("Position") == 1,
                result_time_ms
            )
            .otherwise(
                lit(None).cast("long")
            )
            .alias("race_time_ms"),

            when(
                (col("Position") > 1)
                &
                col("Time").isNotNull(),
                result_time_ms
            )
            .otherwise(
                lit(None).cast("long")
            )
            .alias("gap_to_winner_ms"),

            # ----------------------------------------------
            # Other race attributes
            # ----------------------------------------------

            clean_string("Status")
                .alias("status"),

            col("Points")
                .cast("decimal(6,2)")
                .alias("points"),

            col("Laps")
                .cast("int")
                .alias("laps_completed"),

            # ----------------------------------------------
            # Lineage
            # ----------------------------------------------

            col("_source_file")
                .alias("source_file"),

            col("_source_file_modification_time")
                .alias("source_modified_at"),

            col("_ingested_at")
                .alias("bronze_ingested_at")
        )
    )

    return race_results_df

In [0]:
def validate_race_results_references(df):
    errors = {}

    # ------------------------------------------------------
    # Race FK
    # ------------------------------------------------------

    valid_races = (
        spark.table(SILVER_RACES_TABLE)
        .select(
            "season",
            "round"
        )
        .distinct()
    )

    missing_races = (
        df
        .select(
            "season",
            "round"
        )
        .distinct()
        .join(
            valid_races,
            ["season", "round"],
            "left_anti"
        )
    )

    missing_race_count = missing_races.count()

    if missing_race_count > 0:
        errors["missing_race"] = missing_race_count

    # ------------------------------------------------------
    # Driver FK
    # ------------------------------------------------------

    valid_drivers = (
        spark.table(SILVER_DRIVERS_TABLE)
        .select("driver_id")
        .distinct()
    )

    missing_drivers = (
        df
        .select("driver_id")
        .distinct()
        .join(
            valid_drivers,
            ["driver_id"],
            "left_anti"
        )
    )

    missing_driver_count = missing_drivers.count()

    if missing_driver_count > 0:
        errors["missing_driver"] = missing_driver_count

    # ------------------------------------------------------
    # Constructor FK
    # ------------------------------------------------------

    valid_constructors = (
        spark.table(SILVER_CONSTRUCTORS_TABLE)
        .select("constructor_id")
        .distinct()
    )

    missing_constructors = (
        df
        .select("constructor_id")
        .distinct()
        .join(
            valid_constructors,
            ["constructor_id"],
            "left_anti"
        )
    )

    missing_constructor_count = (
        missing_constructors.count()
    )

    if missing_constructor_count > 0:
        errors["missing_constructor"] = (
            missing_constructor_count
        )

    if errors:
        raise ValueError(
            f"Race results referential validation failed: {errors}"
        )

    print("Race results referential validation OK.")

In [0]:
def validate_race_results(df):
    validation = (
        df
        .agg(
            sum_spark(
                when(
                    col("season").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_season"),

            sum_spark(
                when(
                    col("round").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_round"),

            sum_spark(
                when(
                    col("driver_id").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_driver_id"),

            sum_spark(
                when(
                    col("constructor_id").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_constructor_id"),

            sum_spark(
                when(
                    col("finish_position").isNull()
                    |
                    (col("finish_position") <= 0),
                    1
                ).otherwise(0)
            ).alias("invalid_finish_position"),

            sum_spark(
                when(
                    col("grid_position") < 0,
                    1
                ).otherwise(0)
            ).alias("invalid_grid_position"),

            sum_spark(
                when(
                    col("points") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_points"),

            sum_spark(
                when(
                    col("laps_completed") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_laps"),

            sum_spark(
                when(
                    col("status").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_status"),

            sum_spark(
                when(
                    col("race_time_ms") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_race_time"),

            sum_spark(
                when(
                    col("gap_to_winner_ms") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_gap")
        )
        .first()
        .asDict()
    )

    errors = {
        rule: value or 0
        for rule, value in validation.items()
        if (value or 0) > 0
    }

    duplicate_count = (
        df
        .groupBy(
            "season",
            "round",
            "driver_id"
        )
        .count()
        .filter(
            col("count") > 1
        )
        .count()
    )

    if duplicate_count > 0:
        errors["duplicate_race_driver"] = duplicate_count

    # Sólo puede haber un ganador por carrera
    invalid_winners = (
        df
        .filter(
            col("finish_position") == 1
        )
        .groupBy(
            "season",
            "round"
        )
        .count()
        .filter(
            col("count") != 1
        )
        .count()
    )

    if invalid_winners > 0:
        errors["invalid_winner_count"] = invalid_winners

    if errors:
        raise ValueError(
            f"Silver race_results validation failed: {errors}"
        )

    print(
        f"Validation OK: {df.count()} race results ready for Silver."
    )

In [0]:
def add_race_results_hash(df):
    business_columns = [
        "season",
        "round",
        "driver_id",
        "driver_number",
        "constructor_id",
        "finish_position",
        "classified_position",
        "grid_position",
        "race_time_ms",
        "gap_to_winner_ms",
        "status",
        "points",
        "laps_completed"
    ]

    hash_expression = concat_ws(
        "||",
        *[
            coalesce(
                col(column).cast("string"),
                lit("<NULL>")
            )
            for column in business_columns
        ]
    )

    return (
        df
        .withColumn(
            "record_hash",
            sha2(
                hash_expression,
                256
            )
        )
        .withColumn(
            "silver_updated_at",
            current_timestamp()
        )
    )

In [0]:
def merge_race_results(df):
    if not spark.catalog.tableExists(
        SILVER_RACE_RESULTS_TABLE
    ):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(
                SILVER_RACE_RESULTS_TABLE
            )
        )

        print(
            f"Created {SILVER_RACE_RESULTS_TABLE}"
        )
        return

    target = DeltaTable.forName(
        spark,
        SILVER_RACE_RESULTS_TABLE
    )

    (
        target.alias("target")
        .merge(
            df.alias("source"),
            """
            target.season = source.season
            AND target.round = source.round
            AND target.driver_id = source.driver_id
            """
        )
        .whenMatchedUpdateAll(
            condition="""
                target.record_hash <> source.record_hash
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merged data into {SILVER_RACE_RESULTS_TABLE}"
    )

# Ejecución completa

In [0]:
race_results_source_df = (
    get_latest_race_results_snapshot()
)

race_results_source_df.display()

race_results_df = transform_race_results(
    race_results_source_df
)

validate_race_results_references(
    race_results_df
)

validate_race_results(
    race_results_df
)

race_results_df = add_race_results_hash(
    race_results_df
)

merge_race_results(
    race_results_df
)